# Sycophancy steering-vector extraction — Google Colab

Runs the Phase 1 pipeline on Colab's free GPU: builds contrastive (sycophantic vs. honest) activations of `Qwen/Qwen2.5-1.5B-Instruct`, fits a per-layer logistic-regression probe, extracts the steering direction, and validates it.

**Before you start:** set the runtime to GPU — *Runtime → Change runtime type → Hardware accelerator: GPU (T4 is fine)*.

This avoids touching Colab's preinstalled `torch`/`numpy`: it adds the repo's `src/` to `sys.path` instead of `pip install`-ing the package (whose version pins would upgrade numpy and risk breaking the prebuilt torch). No Hugging Face token is needed (the model is Apache-2.0).

In [ ]:
# 1. Clone the repo (use the feature branch until it is merged to main).
BRANCH = "feat/steering-vector"  # change to "main" once merged
![ -d ml4g-activation-steering ] || git clone --branch $BRANCH --depth 1 https://github.com/frnkly/ml4g-activation-steering.git
%cd ml4g-activation-steering

In [ ]:
# 2. Install only what Colab might be missing. torch/numpy are already present;
#    we do NOT install the package itself so its version pins can't upgrade them.
!pip install -q -U "transformers>=4.45" accelerate scikit-learn matplotlib tqdm certifi

In [ ]:
# 3. Make the package importable and confirm the GPU is visible.
import sys, os
sys.path.insert(0, os.path.abspath("src"))

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — set Runtime → Change runtime type → GPU. CPU works but is slow.")

In [ ]:
# 4. Run the full extraction pipeline. Writes artifacts to ./outputs.
#    First run downloads the model (~3 GB) and extracts activations for 200 pairs.
from syco_steering.pipeline import run_pipeline

metrics = run_pipeline(out_dir="outputs")
metrics

In [ ]:
# 5. Check the acceptance criteria (spec §9).
from syco_steering import config

n_layers = metrics["n_layers"]
best = metrics["best_layer"]
checks = {
    f"test_acc ≥ {config.MIN_TEST_ACC}": metrics["test_acc"] >= config.MIN_TEST_ACC,
    f"auroc ≥ {config.MIN_AUROC}": metrics["auroc"] >= config.MIN_AUROC,
    f"caa_cosine ≥ {config.MIN_CAA_COSINE}": metrics["caa_cosine"] >= config.MIN_CAA_COSINE,
    "best layer in middle third (not 0–2)": n_layers / 3 <= best <= 2 * n_layers / 3,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
print("\nALL PASSED" if all(checks.values()) else "\nSOME CHECKS FAILED — see spec §9 for what to investigate.")

In [ ]:
# 6. Show the diagnostic plots inline.
from IPython.display import Image, display
display(Image("outputs/layer_acc.png"))
display(Image("outputs/projection_hist.png"))

In [ ]:
# 7. Download the artifacts (steering_vector.npz, metrics.json, plots).
import shutil
from google.colab import files

shutil.make_archive("syco_outputs", "zip", "outputs")
files.download("syco_outputs.zip")